# Week4
## Strict Fixed-Baseline-Template Sorting on Compressed-Reconstructed Data

This notebook performs strict sorting on compressed data using baseline templates only.
No baseline spike-time candidate anchoring is used for sorting.


In [12]:
import gc
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from pathlib import Path
from kilosort import template_matching as ks_tm

ROOT = Path('F:/academic')
DATA_ROOT = ROOT / '.test_data'
BASE_RESULTS = DATA_ROOT / 'kilosort4'

compressed_bin = ROOT / 'whitened_recon_ratio_0.10.bin'
if not compressed_bin.exists():
    compressed_bin = ROOT / 'whitened_recon_ratio_0.20.bin'
if not compressed_bin.exists():
    raise FileNotFoundError('Missing whitened_recon_ratio_*.bin')

print('Compressed bin:', compressed_bin)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)


Compressed bin: F:\academic\whitened_recon_ratio_0.10.bin
device: cuda


In [13]:
# Load baseline templates/results from uncompressed data
templates_base = np.load(BASE_RESULTS / 'templates.npy')  # [n_templates, nt, n_chan_tpl]
ops_base = np.load(BASE_RESULTS / 'ops.npy', allow_pickle=True).item()
st_base = np.load(BASE_RESULTS / 'spike_times.npy').squeeze().astype(np.int64)
clu_base = np.load(BASE_RESULTS / 'spike_clusters.npy').squeeze().astype(np.int64)

n_templates, nt, n_chan_tpl = templates_base.shape
print('templates_base shape:', templates_base.shape)

# Project data format: reconstructed compressed data is float32, 383 channels
data_dtype = np.float32
src_n_chan = 383
raw = np.memmap(compressed_bin, dtype=data_dtype, mode='r')
n_samples = raw.size // src_n_chan
data_cmp = raw[: n_samples * src_n_chan].reshape(n_samples, src_n_chan)
n_use = min(n_chan_tpl, src_n_chan)
templates_fixed = templates_base[:, :, :n_use].astype(np.float32, copy=False)
print('compressed data shape:', data_cmp.shape, '| channels used:', n_use)


templates_base shape: (267, 61, 383)
compressed data shape: (1350000, 383) | channels used: 383


### Note: 383 vs 385 channels
Baseline runs may use full probe channel definitions (e.g., including extra/sync/reference channels).
Compressed reconstructed data here has 383 channels; fixed-template matching uses shared channels only.


In [14]:
# Prepare Kilosort fixed-template matching objects on GPU/CPU
for _name in ['U', 'ctc', 'wPCA_t', 'tpl_t', 'X']:
    if _name in globals():
        try:
            del globals()[_name]
        except Exception:
            pass
gc.collect()
if device.type == 'cuda':
    try:
        torch.cuda.empty_cache()
    except Exception:
        pass

ops_tm = dict(ops_base)
ops_tm['settings'] = dict(ops_base.get('settings', {}))
ops_tm['nt'] = int(nt)
ops_tm['nt0min'] = int(ops_base.get('nt0min', nt//2))
ops_tm['Th_learned'] = float(ops_base.get('Th_learned', 8))
ops_tm['max_peels'] = int(ops_base.get('max_peels', 8))

wPCA = ops_base.get('wPCA', None)
if wPCA is None:
    raise RuntimeError('ops_base does not contain wPCA required by template_matching.run_matching')
wPCA_t = torch.from_numpy(np.asarray(wPCA, dtype=np.float32)).to(device=device).contiguous()
ops_tm['wPCA'] = wPCA_t

tpl_t = torch.as_tensor(templates_fixed, dtype=torch.float32, device=device)
# [u,t,c] x [p,t] -> [u,p,c]
U = torch.einsum('utc,pt->upc', tpl_t, wPCA_t).contiguous()
del tpl_t

ctc = ks_tm.prepare_matching(ops_tm, U)
print('wPCA:', tuple(wPCA_t.shape), 'U:', tuple(U.shape), 'ctc:', tuple(ctc.shape))


wPCA: (6, 61) U: (267, 6, 383) ctc: (267, 267, 123)


In [15]:
# Strict sorting on full compressed timeline (no baseline candidate anchoring)
win = 12000   # reduce if OOM
overlap = nt * 2

all_t = []
all_k = []
all_amp = []
all_th = []

for t0 in range(0, n_samples, win):
    t1 = min(n_samples, t0 + win)
    # local context padding for edge-stable matching
    x0 = max(0, t0 - overlap)
    x1 = min(n_samples, t1 + overlap)
    if (x1 - x0) < nt:
        continue

    X_np = np.array(data_cmp[x0:x1, :n_use].T, dtype=np.float32, copy=True)
    X = torch.from_numpy(X_np).to(device=device, non_blocking=True).contiguous()

    with torch.no_grad():
        stt, amps, th_amps, _ = ks_tm.run_matching(ops_tm, X, U, ctc, device=device)

    t_abs = stt[:,0].detach().cpu().numpy().astype(np.int64) + x0
    k_abs = stt[:,1].detach().cpu().numpy().astype(np.int64)
    a_abs = amps.detach().cpu().numpy().squeeze()
    h_abs = th_amps.detach().cpu().numpy().squeeze()

    # keep only center region to avoid duplicate detections across overlapping windows
    keep = (t_abs >= t0) & (t_abs < t1)
    if np.any(keep):
        all_t.append(t_abs[keep])
        all_k.append(k_abs[keep])
        all_amp.append(a_abs[keep])
        all_th.append(h_abs[keep])

    del X
    if device.type == 'cuda':
        torch.cuda.empty_cache()

if len(all_t) == 0:
    st_cmp_fixed = np.empty((0,), dtype=np.int64)
    clu_cmp_fixed = np.empty((0,), dtype=np.int64)
    amp_cmp_fixed = np.empty((0,), dtype=np.float32)
    th_cmp_fixed = np.empty((0,), dtype=np.float32)
else:
    st_cmp_fixed = np.concatenate(all_t)
    clu_cmp_fixed = np.concatenate(all_k)
    amp_cmp_fixed = np.concatenate(all_amp)
    th_cmp_fixed = np.concatenate(all_th)

    # global time sort
    order = np.argsort(st_cmp_fixed)
    st_cmp_fixed = st_cmp_fixed[order]
    clu_cmp_fixed = clu_cmp_fixed[order]
    amp_cmp_fixed = amp_cmp_fixed[order]
    th_cmp_fixed = th_cmp_fixed[order]

print('strict sorted spikes:', st_cmp_fixed.shape[0])
print('unique predicted templates:', np.unique(clu_cmp_fixed).shape[0] if st_cmp_fixed.size else 0)


strict sorted spikes: 123348
unique predicted templates: 265


In [16]:
# Save strict fixed-template sort output
out = {
    'compressed_bin': str(compressed_bin),
    'method': 'strict_kilosort_run_matching_fixed_baseline_templates_full_timeline',
    'st': st_cmp_fixed,
    'clu': clu_cmp_fixed,
    'amp': amp_cmp_fixed,
    'th_amp': th_cmp_fixed,
    'template_shape': templates_fixed.shape,
    'compressed_shape': data_cmp.shape,
}
np.save(ROOT / 'week4_kilosort_function_fixed_template_results.npy', out, allow_pickle=True)
print('saved:', ROOT / 'week4_kilosort_function_fixed_template_results.npy')


saved: F:\academic\week4_kilosort_function_fixed_template_results.npy


## Result Analysis (Evaluation Only)
Baseline data is used only for evaluation alignment, not for candidate selection during sorting.


In [ ]:
# Time-alignment evaluation: strict sort result -> nearest baseline spike
tol = 12
baseline_st = st_base.astype(np.int64)
baseline_clu = clu_base.astype(np.int64)
test_st = st_cmp_fixed.astype(np.int64)
test_clu = clu_cmp_fixed.astype(np.int64)

if test_st.size == 0 or baseline_st.size == 0:
    labels = np.empty((0,), dtype=np.int64)
    matched = np.empty((0,), dtype=bool)
else:
    pos = np.searchsorted(baseline_st, test_st)
    left = np.clip(pos-1, 0, baseline_st.size-1)
    right = np.clip(pos, 0, baseline_st.size-1)
    dl = np.abs(test_st - baseline_st[left])
    dr = np.abs(test_st - baseline_st[right])
    use_l = dl <= dr
    nn = np.where(use_l, left, right)
    dt = np.abs(test_st - baseline_st[nn])
    matched = dt <= tol
    labels = np.full(test_st.shape[0], -1, dtype=np.int64)
    labels[matched] = baseline_clu[nn[matched]]

coverage_vs_baseline = matched.mean() if matched.size else 0.0
print(f'Matched to baseline within +/-{tol} samples: {matched.sum()} / {matched.size} ({coverage_vs_baseline*100:.2f}%)')

if matched.any():
    acc = (test_clu[matched] == labels[matched]).mean()
    print(f'Label agreement on matched spikes: {acc*100:.2f}%')
else:
    print('No matched spikes for label comparison.')


In [ ]:
# Cluster-level summary on matched spikes
if matched.size == 0:
    summary = pd.DataFrame(columns=['baseline_clu','n_ref','n_matched_from_test','recall_from_test_side','same_label_rate'])
else:
    dfm = pd.DataFrame({
        'test_st': test_st,
        'test_clu': test_clu,
        'baseline_clu': labels,
        'matched': matched,
    })

    ref_counts = pd.Series(baseline_clu).value_counts().sort_index()
    m = dfm[dfm['matched']].copy()
    g = m.groupby('baseline_clu')
    n_matched = g.size()
    same_rate = g.apply(lambda x: (x['test_clu'].values == x['baseline_clu'].values).mean())

    summary = pd.DataFrame({
        'baseline_clu': ref_counts.index.values,
        'n_ref': ref_counts.values,
    })
    summary = summary.merge(n_matched.rename('n_matched_from_test'), left_on='baseline_clu', right_index=True, how='left')
    summary = summary.merge(same_rate.rename('same_label_rate'), left_on='baseline_clu', right_index=True, how='left')
    summary['n_matched_from_test'] = summary['n_matched_from_test'].fillna(0).astype(int)
    summary['recall_from_test_side'] = summary['n_matched_from_test'] / summary['n_ref']

display(summary.sort_values('n_ref', ascending=False).head(20))


In [ ]:
# Distributions and export
fig, ax = plt.subplots(1, 3, figsize=(15, 4))

if st_cmp_fixed.size:
    ax[0].hist(clu_cmp_fixed, bins=80)
ax[0].set_title('Predicted template IDs (strict sort)')
ax[0].set_xlabel('template id')
ax[0].set_ylabel('count')

amp_ok = np.isfinite(amp_cmp_fixed)
if np.any(amp_ok):
    ax[1].hist(amp_cmp_fixed[amp_ok], bins=80)
ax[1].set_title('Amplitude distribution')

th_ok = np.isfinite(th_cmp_fixed)
if np.any(th_ok):
    ax[2].hist(th_cmp_fixed[th_ok], bins=80)
ax[2].set_title('Threshold amplitude distribution')

plt.tight_layout()
plt.show()

summary.to_csv(ROOT / 'week4_fixed_template_cluster_summary.csv', index=False)
print('saved:', ROOT / 'week4_fixed_template_cluster_summary.csv')
